In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

epochs = 4
batch_size = 4
learning_rate = 0.001


In [4]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5,0.5,0.5))])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

classes = ('plane', 'car', 'bird', 'cat', 
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100.0%
c:\Users\gkwock\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [8]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
    
    def forward(self, x):
        #first convolution
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        #Second convolution
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
        #flatten
        x = x.view(-1, 16*5*5)
        
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        return x
        
        
    
    
model = ConvNet().to(device)

criterion = nn.CrossEntropyLoss() # has Softmax function at the end
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [9]:
n_total_steps = len(train_loader)
for epoch in range(epochs):
    for i, (images, labels) in enumerate(train_loader):
        
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 2000 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Step [{i+1}/{n_total_steps}. Loss: {loss.item()}]')
            
print('Finished Training')

Epoch [1/4], Step [2000/12500. Loss: 2.2681198120117188]
Epoch [1/4], Step [4000/12500. Loss: 2.29128360748291]
Epoch [1/4], Step [6000/12500. Loss: 2.282886505126953]
Epoch [1/4], Step [8000/12500. Loss: 2.2513463497161865]
Epoch [1/4], Step [10000/12500. Loss: 2.0338897705078125]
Epoch [1/4], Step [12000/12500. Loss: 1.988586664199829]
Epoch [2/4], Step [2000/12500. Loss: 2.1803717613220215]
Epoch [2/4], Step [4000/12500. Loss: 1.9984068870544434]
Epoch [2/4], Step [6000/12500. Loss: 1.875053882598877]
Epoch [2/4], Step [8000/12500. Loss: 2.359034299850464]
Epoch [2/4], Step [10000/12500. Loss: 1.0903563499450684]
Epoch [2/4], Step [12000/12500. Loss: 2.116990327835083]
Epoch [3/4], Step [2000/12500. Loss: 1.7780570983886719]
Epoch [3/4], Step [4000/12500. Loss: 1.5150375366210938]
Epoch [3/4], Step [6000/12500. Loss: 1.9903749227523804]
Epoch [3/4], Step [8000/12500. Loss: 1.3071902990341187]
Epoch [3/4], Step [10000/12500. Loss: 3.0469985008239746]
Epoch [3/4], Step [12000/12500. L

In [ ]:
# evaluating
with torch.no_grad():
    n_correct = 0
    n_samples = 0
    n_class_correct = [0 for i in range(10)]
    n_class_samples = [0 for i in range(10)]
    
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        
        _, predicted = torch.max(outputs, 1)
        n_samples += labels.size(0)
        n_correct += (predicted == labels).sum().item()
        
        for i in range(batch_size):
            label = labels[i]
            pred = predicted[i]
            if (label == pred):
                n_class_correct[label] +=1
            n_class_samples[label] +=1
            
    acc = 100.0 * n_correct / n_samples
    print(f'Accuracy: {acc} %')
        
    for i in range(10):
        if(n_class_samples[i] > 0):
            acc = 100.0 * n_class_correct[i] / n_class_samples[i]
            print(f'Accuracy of {classes[i]}: {acc} %')
            

Accuracy: 45.01 %
[493, 737, 255, 208, 310, 319, 546, 635, 592, 406]
[1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
Accuracy of plane: 49.3 %
Accuracy of car: 73.7 %
Accuracy of bird: 25.5 %
Accuracy of cat: 20.8 %
Accuracy of deer: 31.0 %
Accuracy of dog: 31.9 %
Accuracy of frog: 54.6 %
Accuracy of horse: 63.5 %
Accuracy of ship: 59.2 %
Accuracy of truck: 40.6 %
